In [ ]:
from IPython.core.display import display_png, Image
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
# 导入工具节点
from langgraph.prebuilt import ToolNode, tools_condition


# ====== Step 1: 准备工具 ======
# 工具
@tool
def get_weather(city: str) -> str:
    """获取城市天气"""
    weather_data = {
        "北京": "晴天 25度",
        "上海": "多云 28度",
        "杭州": "小雨 22度",
    }
    return weather_data.get(city, f"未找到{city}的天气信息")

# 组成工具数组
tools = [get_weather]

# ====== Step 2: 准备模型，绑定工具 ======
# 定义模型
model = init_chat_model(
    'deepseek-v4-flash',
    extra_body = {'thinking': {'type': 'disabled'}}
)
# 绑定工具到模型
# 高速大模型有哪些工具
model_with_tools = model.bind_tools(tools)

# ====== Step 3: 定义Node ======
def llm_node(state: MessagesState):
    """LLM节点：调用模型，可以选择调用工具"""
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# ====== Step 4: 构建图 ======
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("llm", llm_node)

# 工具节点函数会被批量添加进工作流中
# 这里是真正的函数,大模型绑定工具只是字符串函数名
graph_builder.add_node("tools", ToolNode(tools))  # 预定义工具节点

# 连线
graph_builder.add_edge(START, "llm")
# 大模型节点到工具节点路由,判断最后一条消息是否为工具调用消息
graph_builder.add_conditional_edges("llm", tools_condition)  # 预定义工具执行路由节点
# 如果是,则由工具节点返回回到大模型节点,再次输出
graph_builder.add_edge("tools", "llm")  # 工具执行后回到LLM，形成循环

# 获得图
agent_graph = graph_builder.compile()

display_png(Image(agent_graph.get_graph().draw_mermaid_png()))

# 阻塞式调用
result = agent_graph.invoke({
    "messages": [HumanMessage(content="北京和杭州今天天气怎么样？")]
})

# 循环打印结果,非流式的话不是迭代器对象,无所谓
for m in result['messages']:
    m.pretty_print()